In [ ]:
from pathlib import Path
import sys
import os
import random

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "src").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "code"))

from import_libraries import *
import common_libraries.generic_library as gn_l

from ml_classification.ml_classification_pipeline import (
    GeospatialClassificationPipeline,
    ClassificationPipelineConfig,
)

# Global reproducibility seed
random_seed = 42
os.environ["PYTHONHASHSEED"] = str(random_seed)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(random_seed)
np.random.seed(random_seed)
try:
    import tensorflow as tf
    tf.random.set_seed(random_seed)
except Exception:
    print("TensorFlow not available, skipping TensorFlow seed setting.")

In [ ]:
# sample_gdf_file = "C:/work_dir/sentinel2_monthly_parcel_statistics_local4/parcels_with_statistics.gpkg"
# gdf_sample = gn_l.read_data(sample_gdf_file)
# gdf_sample = gdf_sample.set_crs("EPSG:2100", allow_override=True)
# gdf_sample = gdf_sample[gdf_sample['label'].notna()].reset_index(drop=True)
# # gdf_sample = gdf_sample.sample(frac=0.2, random_state=42).reset_index(drop=True)
# gdf_train = gdf_sample.copy()

# unknown_fraction = 0.20
# rng = np.random.default_rng(random_seed)
# unknown_count = int(len(gdf_train) * unknown_fraction)
# unknown_indices = rng.choice(gdf_train.index.to_numpy(), size=unknown_count, replace=False)
# gdf_train.loc[unknown_indices, "label"] = None

# features = [col for col in gdf_sample.columns if col not in ["label", "geometry"]]

# print(f"Sampled rows: {len(gdf_sample)}")
# print(f"Unknown labels in training data: {gdf_train['label'].isna().sum()}")
# display(gdf_train.head())
# display(gdf_sample.head())

In [ ]:
project_dir = "C:/work_dir/test_new_ml"

import common_libraries.io_library as io_l
gdf_train = "C:/work_dir/sentinel2_monthly_parcel_statistics_local4/parcels_with_statistics.gpkg"
gdf_train = io_l.read_data(gdf_train)
gdf_train = gdf_train.set_crs("EPSG:2100", allow_override=True)

import common_libraries.geom_library as geom_l
gdf_train = geom_l.convert_geometries_to_points(gdf_train)

In [ ]:
features = [col for col in gdf_train.columns if col not in ["class", "period_end", "geometry"]]

In [ ]:
config = ClassificationPipelineConfig(
    project_dir=project_dir,
    target_column="class",
    feature_columns=features,
    id_column="parcel_id",
    cv_folds=5,
    test_size=0.2,
    random_state=random_seed,
    n_jobs=1,
    max_search_candidates=12,
    selection_type="soft_voting",
    interpretability_top_models=3,
    interpretability_include_shap=False,    
    top_voting_models=None,
    # label_balancing_method="random_oversample",
    spatial_split=True,
    spatial_split_grid_size=10,
    selected_models=(
        "gradient_boosting",
        "hist_gradient_boosting",
        "bagging",
        "random_forest",
        "decision_tree",
        "extra_trees",
        "bayesian",
        "xgboost",
        #"lightgbm",
        "knn",
        "neural_network"
   ),
    max_map_geometries=20000,
    open_html_report=True,
)
pipeline = GeospatialClassificationPipeline(config)

In [ ]:
summary_check = pipeline.run_check(gdf_train)
summary_check

In [ ]:
summary_prepare = pipeline.run_prepare()

In [ ]:
summary_train = pipeline.run_train()
summary_train

In [ ]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict

In [ ]:
# Show only rows that were unknown during training
classified_df = pd.read_csv(os.path.join(project_dir, "05_predict", "final_predictions_preview.csv"))

# # merge based on index
classified_gdf = pd.merge(gdf_sample, classified_df, left_index=True, right_on="row_id", how="left")

unknown_predictions = classified_gdf.loc[gdf_train["label"].isna()].copy()
display(unknown_predictions["label_prediction"].value_counts(dropna=False).rename_axis("predicted_label").reset_index(name="count"))

In [ ]:
classified_gdf["label_prediction"] = classified_gdf["label_prediction"].astype("str")
classified_gdf["label_x"] = classified_gdf["label_x"].astype("str")

gdf_result = classified_gdf[classified_gdf["label_y"].isna()].copy()
gdf_result["accuracy"] = gdf_result["label_prediction"] == gdf_result["label_x"]

accuracy = gdf_result["accuracy"].mean()
print(f"Accuracy on unknown data: {accuracy:.2%}")